In [25]:
import csv
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple

import cv2
from ultralytics import YOLO

In [26]:

# -----------------------------
# PATHS
# -----------------------------
PROJECT_ROOT = Path(r"C:\Users\benne\Computer_Vision\BoxingPunchDetection")
FINAL_MODEL_PATH = PROJECT_ROOT / "yolo26_models" / "resume_yolo26s" / "weights" / "best.pt"
VIDEO_PATH = PROJECT_ROOT / "input_video/task_kam4_gh199681/data/GH199681.mp4"
ANNOTATIONS_PATH = PROJECT_ROOT / "input_video/task_kam4_gh199681/annotations.json"

OUTPUT_ROOT = Path("runs")
RUN_TAG = "26s_bgs"
RUN_NAME = f"{VIDEO_PATH.stem}_{RUN_TAG}"
RUN_DIR = OUTPUT_ROOT / RUN_NAME

# -----------------------------
# TOGGLES
# -----------------------------
COMPARE_WITH_ANNOTATIONS = True
SAVE_ANNOTATED_VIDEO = True
SHOW_VIDEO = False
CONF = 0.4
IOU = 0.5
MAX_DET = 2
COMPARISON_IOU_THRESHOLD = 0.5
MAX_FRAMES: Optional[int] = None  # set to an int for quick testing, e.g. 500

# -----------------------------
# TEMPORAL FILTER TOGGLE
# -----------------------------
USE_TEMPORAL_FILTER = True
TEMPORAL_IOU = 0.3
TEMPORAL_WINDOW = 3
MIN_TEMPORAL_HITS = 2

USE_TRACK_LENGTH_FILTER = False
TRACK_IOU = 0.3
MIN_TRACK_LENGTH = 5

# -----------------------------
# FAST BACKGROUND SUBTRACTION / MOTION GATING
# -----------------------------
USE_BACKGROUND_SUBTRACTION = True
BACKGROUND_SUBTRACTION_MODE = "gate"  # "gate", "apply", or "mask"

# gate = fastest recommended mode: use background subtraction only to decide
# whether YOLO should run. YOLO receives the original frame.
# apply/mask = YOLO receives a foreground-only frame, but this may hurt accuracy.
BG_FEED_YOLO_FOREGROUND = False

BG_HISTORY = 120              # smaller than 500 = faster adaptation
BG_VAR_THRESHOLD = 16
BG_DETECT_SHADOWS = False
BG_LEARNING_RATE = -1
BG_WARMUP_FRAMES = 30

# Biggest speed knob. 0.50 means 1/4 pixels. 0.35 means ~1/8 pixels.
BG_SCALE = 0.35

# Count foreground pixels on the small mask. Raise if too many frames pass.
# Lower if YOLO skips punches.
BG_MOTION_PIXEL_THRESHOLD = 900

# Run YOLO occasionally even when idle so you do not completely miss slow motion.
# Set to None or 0 to disable idle checks.
YOLO_EVERY_N_FRAMES_WHEN_IDLE: Optional[int] = 20

# Morphology is expensive. Keep this small or set to 0/1 to disable.
BG_MORPH_KERNEL_SIZE = 3
BG_DILATE_ITERATIONS = 1

# If True, show a few progress lines during long videos.
PRINT_PROGRESS_EVERY = 250


In [27]:
# -----------------------------
# CLASS MAPS
# -----------------------------
CLASS_ID_TO_NAME = {
    0: "left_head",
    1: "right_head",
    2: "left_body",
    3: "right_body",
    4: "left_block",
    5: "right_block",
    6: "left_miss",
    7: "right_miss",
}

POLISH_TO_NAME = {
    "Głowa lewą ręką": "left_head",
    "Głowa prawą ręką": "right_head",
    "Korpus lewą ręką": "left_body",
    "Korpus prawą ręką": "right_body",
    "Blok lewą ręką": "left_block",
    "Blok prawą ręką": "right_block",
    "Chybienie lewą ręką": "left_miss",
    "Chybienie prawą ręką": "right_miss",
}


@dataclass
class BoxRecord:
    frame: int
    cls_name: str
    x1: float
    y1: float
    x2: float
    y2: float
    conf: Optional[float] = None

In [28]:
def ensure_exists(path: Path, name: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")


def get_video_info(video_path: Path) -> Dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    info = {
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return info


def save_json(path: Path, data) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def make_bg_kernel():
    if BG_MORPH_KERNEL_SIZE and BG_MORPH_KERNEL_SIZE > 1:
        return cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (BG_MORPH_KERNEL_SIZE, BG_MORPH_KERNEL_SIZE),
        )
    return None


def fast_motion_frame_generator(video_path: Path) -> Iterator[Tuple[int, object, int, str]]:
    """Yield selected frames for YOLO.

    Returns tuples: (original_frame_index, yolo_frame, motion_pixels, reason)
    The foreground mask is computed on a downscaled frame for speed.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    subtractor = cv2.createBackgroundSubtractorMOG2(
        history=BG_HISTORY,
        varThreshold=BG_VAR_THRESHOLD,
        detectShadows=BG_DETECT_SHADOWS,
    )
    kernel = make_bg_kernel()

    frame_idx = 0
    yielded = 0
    skipped = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
                break

            small = cv2.resize(frame, None, fx=BG_SCALE, fy=BG_SCALE, interpolation=cv2.INTER_AREA)
            fg_mask = subtractor.apply(small, learningRate=BG_LEARNING_RATE)

            if frame_idx < BG_WARMUP_FRAMES:
                frame_idx += 1
                continue

            # MOG2 already returns a binary-ish mask when shadows are disabled.
            # Thresholding keeps behavior consistent and countNonZero is much
            # faster than finding contours.
            _, fg_mask = cv2.threshold(fg_mask, 127, 255, cv2.THRESH_BINARY)

            if kernel is not None:
                fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
                if BG_DILATE_ITERATIONS > 0:
                    fg_mask = cv2.dilate(fg_mask, kernel, iterations=BG_DILATE_ITERATIONS)

            motion_pixels = int(cv2.countNonZero(fg_mask))
            has_motion = motion_pixels >= BG_MOTION_PIXEL_THRESHOLD
            idle_check = bool(
                YOLO_EVERY_N_FRAMES_WHEN_IDLE
                and frame_idx % YOLO_EVERY_N_FRAMES_WHEN_IDLE == 0
            )

            if has_motion or idle_check:
                reason = "motion" if has_motion else "idle_check"
                if BG_FEED_YOLO_FOREGROUND or BACKGROUND_SUBTRACTION_MODE in {"apply", "mask"}:
                    full_mask = cv2.resize(
                        fg_mask,
                        (frame.shape[1], frame.shape[0]),
                        interpolation=cv2.INTER_NEAREST,
                    )
                    if BACKGROUND_SUBTRACTION_MODE == "mask":
                        yolo_frame = cv2.cvtColor(full_mask, cv2.COLOR_GRAY2BGR)
                    else:
                        yolo_frame = cv2.bitwise_and(frame, frame, mask=full_mask)
                else:
                    yolo_frame = frame

                yielded += 1
                yield frame_idx, yolo_frame, motion_pixels, reason
            else:
                skipped += 1

            if PRINT_PROGRESS_EVERY and frame_idx > 0 and frame_idx % PRINT_PROGRESS_EVERY == 0:
                print(
                    f"BG gate frame={frame_idx} yielded={yielded} skipped={skipped} "
                    f"motion_pixels={motion_pixels}"
                )

            frame_idx += 1
    finally:
        cap.release()


def iou_xyxy(a: BoxRecord, b: BoxRecord) -> float:
    inter_x1 = max(a.x1, b.x1)
    inter_y1 = max(a.y1, b.y1)
    inter_x2 = min(a.x2, b.x2)
    inter_y2 = min(a.y2, b.y2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(0.0, a.x2 - a.x1) * max(0.0, a.y2 - a.y1)
    area_b = max(0.0, b.x2 - b.x1) * max(0.0, b.y2 - b.y1)
    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


def draw_box_record(frame, box: BoxRecord) -> None:
    x1, y1, x2, y2 = map(int, [box.x1, box.y1, box.x2, box.y2])
    label = box.cls_name if box.conf is None else f"{box.cls_name} {box.conf:.2f}"
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    text_size, baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    text_w, text_h = text_size
    label_y1 = max(0, y1 - text_h - baseline - 4)
    label_y2 = max(text_h + baseline + 4, y1)
    cv2.rectangle(frame, (x1, label_y1), (x1 + text_w + 6, label_y2), (0, 255, 0), thickness=-1)
    cv2.putText(frame, label, (x1 + 3, label_y2 - baseline - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)


def write_annotated_video_on_original(video_path: Path, pred_by_frame: Dict[int, List[BoxRecord]], output_path: Path) -> Path:
    ensure_exists(video_path, "Video")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    if not writer.isOpened():
        cap.release()
        raise ValueError(f"Could not create video writer: {output_path}")
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        for box in pred_by_frame.get(frame_idx, []):
            draw_box_record(frame, box)
        writer.write(frame)
        frame_idx += 1
        if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
            break
    cap.release()
    writer.release()
    return output_path

In [29]:
def load_ground_truth(annotations_path: Path) -> Dict[int, List[BoxRecord]]:
    with open(annotations_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        if not data:
            raise ValueError("annotations.json is empty")
        data = data[0]
    gt_by_frame: Dict[int, List[BoxRecord]] = {}
    for track in data.get("tracks", []):
        cls_name = POLISH_TO_NAME.get(track.get("label"))
        if cls_name is None:
            continue
        for shape in track.get("shapes", []):
            if shape.get("type") != "rectangle" or shape.get("outside", False):
                continue
            points = shape.get("points", [])
            frame_idx = shape.get("frame")
            if frame_idx is None or len(points) != 4:
                continue
            x1, y1, x2, y2 = points
            gt_by_frame.setdefault(int(frame_idx), []).append(
                BoxRecord(int(frame_idx), cls_name, float(x1), float(y1), float(x2), float(y2), None)
            )
    return gt_by_frame


In [30]:
def run_model(model_path: Path, video_path: Path, run_dir: Path) -> Tuple[Optional[Path], Dict[int, List[BoxRecord]], Dict]:
    run_dir.mkdir(parents=True, exist_ok=True)
    ensure_exists(model_path, "Model")
    ensure_exists(video_path, "Video")

    model = YOLO(model_path)
    print(f"Using model: {model_path}")
    print(f"Testing video: {video_path}")
    print(f"Saving outputs to: {run_dir}")

    prediction_records: Dict[int, List[BoxRecord]] = {}
    prediction_json_frames = []
    processed_frames = 0
    selected_frames = 0
    skipped_by_gate = 0

    if USE_BACKGROUND_SUBTRACTION:
        total_frames = get_video_info(video_path).get("frame_count", 0)
        last_seen_frame = -1
        frame_source = fast_motion_frame_generator(video_path)
        for frame_idx, frame, motion_pixels, reason in frame_source:
            skipped_by_gate += max(0, frame_idx - last_seen_frame - 1)
            last_seen_frame = frame_idx
            result = model.predict(
                source=frame,
                save=False,
                conf=CONF,
                iou=IOU,
                max_det=MAX_DET,
                verbose=False,
            )[0]
            selected_frames += 1
            frame_boxes, frame_entries = result_to_records(result, frame_idx)
            prediction_records[frame_idx] = frame_boxes
            prediction_json_frames.append({
                "frame": frame_idx,
                "motion_pixels": motion_pixels,
                "gate_reason": reason,
                "detections": frame_entries,
            })
            processed_frames += 1
        if total_frames:
            skipped_by_gate = max(0, min(total_frames, MAX_FRAMES or total_frames) - selected_frames - BG_WARMUP_FRAMES)
    else:
        results_iter = model.predict(
            source=str(video_path),
            save=False,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            show=SHOW_VIDEO,
            stream=True,
            line_width=3,
            verbose=False,
        )
        for frame_idx, result in enumerate(results_iter):
            frame_boxes, frame_entries = result_to_records(result, frame_idx)
            prediction_records[frame_idx] = frame_boxes
            prediction_json_frames.append({"frame": frame_idx, "detections": frame_entries})
            processed_frames += 1
            if MAX_FRAMES is not None and processed_frames >= MAX_FRAMES:
                break

    predictions_json = {
        "model_path": str(model_path),
        "video_path": str(video_path),
        "frames_processed_by_yolo": processed_frames,
        "selected_frames": selected_frames if USE_BACKGROUND_SUBTRACTION else processed_frames,
        "skipped_by_bg_gate_estimate": skipped_by_gate,
        "confidence_threshold": CONF,
        "iou_threshold": IOU,
        "background_subtraction_used": USE_BACKGROUND_SUBTRACTION,
        "background_subtraction_mode": BACKGROUND_SUBTRACTION_MODE,
        "bg_scale": BG_SCALE,
        "bg_motion_pixel_threshold": BG_MOTION_PIXEL_THRESHOLD,
        "predictions": prediction_json_frames,
    }
    save_json(run_dir / "predictions.json", predictions_json)

    with open(run_dir / "predictions_flat.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "class_name", "confidence", "x1", "y1", "x2", "y2"])
        for frame_idx in sorted(prediction_records.keys()):
            for box in prediction_records[frame_idx]:
                writer.writerow([frame_idx, box.cls_name, box.conf, box.x1, box.y1, box.x2, box.y2])

    summary = {
        "predicted_video": None,
        "predictions_json": str(run_dir / "predictions.json"),
        "predictions_csv": str(run_dir / "predictions_flat.csv"),
        "frames_processed_by_yolo": processed_frames,
        "background_subtraction_used": USE_BACKGROUND_SUBTRACTION,
        "bg_scale": BG_SCALE,
        "bg_motion_pixel_threshold": BG_MOTION_PIXEL_THRESHOLD,
    }
    return None, prediction_records, summary


def result_to_records(result, frame_idx: int) -> Tuple[List[BoxRecord], List[Dict]]:
    frame_entries = []
    frame_boxes: List[BoxRecord] = []
    if result.boxes is not None and len(result.boxes) > 0:
        xyxy = result.boxes.xyxy.cpu().tolist()
        confs = result.boxes.conf.cpu().tolist()
        clss = result.boxes.cls.cpu().tolist()
        for box_xyxy, conf, cls_id in zip(xyxy, confs, clss):
            cls_id = int(cls_id)
            cls_name = CLASS_ID_TO_NAME.get(cls_id, str(cls_id))
            x1, y1, x2, y2 = [float(v) for v in box_xyxy]
            rec = BoxRecord(frame_idx, cls_name, x1, y1, x2, y2, float(conf))
            frame_boxes.append(rec)
            frame_entries.append({
                "class_id": cls_id,
                "class_name": cls_name,
                "confidence": float(conf),
                "xyxy": [x1, y1, x2, y2],
            })
    return frame_boxes, frame_entries

In [31]:
def temporal_filter_predictions(pred_by_frame: Dict[int, List[BoxRecord]], temporal_iou: float = 0.3, temporal_window: int = 2, min_hits: int = 2) -> Dict[int, List[BoxRecord]]:
    filtered: Dict[int, List[BoxRecord]] = {}
    for frame_idx, preds in pred_by_frame.items():
        kept = []
        for pred in preds:
            hits = 1
            for other_frame in range(frame_idx - temporal_window, frame_idx + temporal_window + 1):
                if other_frame == frame_idx:
                    continue
                for other in pred_by_frame.get(other_frame, []):
                    if other.cls_name == pred.cls_name and iou_xyxy(pred, other) >= temporal_iou:
                        hits += 1
                        break
                if hits >= min_hits:
                    break
            if hits >= min_hits:
                kept.append(pred)
        filtered[frame_idx] = kept
    return filtered


def filter_short_tracks(pred_by_frame: Dict[int, List[BoxRecord]], track_iou: float = 0.3, min_track_length: int = 5) -> Dict[int, List[BoxRecord]]:
    frames = sorted(pred_by_frame.keys())
    tracks = []
    for frame_idx in frames:
        for pred in pred_by_frame[frame_idx]:
            best_track = None
            best_iou = 0
            for track in tracks:
                last_frame, last_pred = track[-1]
                if frame_idx - last_frame > 1 or pred.cls_name != last_pred.cls_name:
                    continue
                score = iou_xyxy(pred, last_pred)
                if score > best_iou:
                    best_iou = score
                    best_track = track
            if best_track is not None and best_iou >= track_iou:
                best_track.append((frame_idx, pred))
            else:
                tracks.append([(frame_idx, pred)])
    filtered: Dict[int, List[BoxRecord]] = {}
    for track in tracks:
        if len(track) >= min_track_length:
            for frame_idx, pred in track:
                filtered.setdefault(frame_idx, []).append(pred)
    return filtered


def compare_predictions_to_gt(pred_by_frame: Dict[int, List[BoxRecord]], gt_by_frame: Dict[int, List[BoxRecord]], run_dir: Path, iou_threshold: float) -> Dict:
    run_dir.mkdir(parents=True, exist_ok=True)
    all_frames = sorted(set(pred_by_frame.keys()) | set(gt_by_frame.keys()))
    total_tp = total_fp = total_fn = 0
    per_frame_rows = []
    match_rows = []
    for frame_idx in all_frames:
        preds = pred_by_frame.get(frame_idx, [])
        gts = gt_by_frame.get(frame_idx, [])
        matched_pred = set()
        matched_gt = set()
        candidate_matches = []
        for pi, pred in enumerate(preds):
            for gi, gt in enumerate(gts):
                if pred.cls_name != gt.cls_name:
                    continue
                score = iou_xyxy(pred, gt)
                if score >= iou_threshold:
                    candidate_matches.append((score, pi, gi))
        candidate_matches.sort(reverse=True, key=lambda x: x[0])
        for score, pi, gi in candidate_matches:
            if pi in matched_pred or gi in matched_gt:
                continue
            matched_pred.add(pi)
            matched_gt.add(gi)
            total_tp += 1
            match_rows.append([frame_idx, preds[pi].cls_name, round(score, 6), preds[pi].conf, preds[pi].x1, preds[pi].y1, preds[pi].x2, preds[pi].y2, gts[gi].x1, gts[gi].y1, gts[gi].x2, gts[gi].y2])
        frame_fp = len(preds) - len(matched_pred)
        frame_fn = len(gts) - len(matched_gt)
        total_fp += frame_fp
        total_fn += frame_fn
        per_frame_rows.append([frame_idx, len(preds), len(gts), len(matched_pred), frame_fp, frame_fn])

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    with open(run_dir / "comparison_per_frame.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "num_predictions", "num_ground_truth", "true_positives", "false_positives", "false_negatives"])
        writer.writerows(per_frame_rows)
    with open(run_dir / "matched_detections.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "class_name", "iou", "pred_confidence", "pred_x1", "pred_y1", "pred_x2", "pred_y2", "gt_x1", "gt_y1", "gt_x2", "gt_y2"])
        writer.writerows(match_rows)

    summary = {
        "iou_threshold": iou_threshold,
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "comparison_per_frame_csv": str(run_dir / "comparison_per_frame.csv"),
        "matched_detections_csv": str(run_dir / "matched_detections.csv"),
    }
    save_json(run_dir / "comparison_summary.json", summary)
    return summary

In [32]:
def main() -> None:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving to: {RUN_DIR}")

    predicted_video, pred_by_frame, pred_summary = run_model(FINAL_MODEL_PATH, VIDEO_PATH, RUN_DIR)
    raw_count = sum(len(v) for v in pred_by_frame.values())

    if USE_TEMPORAL_FILTER:
        pred_by_frame = temporal_filter_predictions(pred_by_frame, TEMPORAL_IOU, TEMPORAL_WINDOW, MIN_TEMPORAL_HITS)

    before_tracks = sum(len(v) for v in pred_by_frame.values())
    if USE_TRACK_LENGTH_FILTER:
        pred_by_frame = filter_short_tracks(pred_by_frame, TRACK_IOU, MIN_TRACK_LENGTH)
    after_tracks = sum(len(v) for v in pred_by_frame.values())

    print(f"Raw predictions: {raw_count}")
    print(f"Before track-length filter: {before_tracks}")
    print(f"After track-length filter: {after_tracks}")

    if SAVE_ANNOTATED_VIDEO:
        predicted_video = write_annotated_video_on_original(
            video_path=VIDEO_PATH,
            pred_by_frame=pred_by_frame,
            output_path=RUN_DIR / f"{VIDEO_PATH.stem}_annotated_original.mp4",
        )
        pred_summary["predicted_video"] = str(predicted_video)
        pred_summary["annotated_on"] = "original_video"
        print(f"Annotated original video saved to: {predicted_video}")

    final_summary = {
        "model_path": str(FINAL_MODEL_PATH),
        "original_video_path": str(VIDEO_PATH),
        "background_subtraction": {
            "enabled": USE_BACKGROUND_SUBTRACTION,
            "mode": BACKGROUND_SUBTRACTION_MODE,
            "bg_scale": BG_SCALE,
            "motion_pixel_threshold": BG_MOTION_PIXEL_THRESHOLD,
            "idle_check_every_n_frames": YOLO_EVERY_N_FRAMES_WHEN_IDLE,
            "feed_yolo_foreground": BG_FEED_YOLO_FOREGROUND,
        },
        "compare_with_annotations": COMPARE_WITH_ANNOTATIONS,
        "prediction_outputs": pred_summary,
    }

    if COMPARE_WITH_ANNOTATIONS:
        ensure_exists(ANNOTATIONS_PATH, "Annotations")
        gt_by_frame = load_ground_truth(ANNOTATIONS_PATH)
        comparison_summary = compare_predictions_to_gt(pred_by_frame, gt_by_frame, RUN_DIR, COMPARISON_IOU_THRESHOLD)
        final_summary["comparison_outputs"] = comparison_summary
        print("Comparison complete.")
        print(json.dumps(comparison_summary, indent=2))
    else:
        print("Comparison skipped because COMPARE_WITH_ANNOTATIONS = False")

    save_json(RUN_DIR / "run_summary.json", final_summary)
    print("\nDone.")
    if predicted_video is not None:
        print(f"Annotated video: {predicted_video}")
    print(f"Predictions JSON: {RUN_DIR / 'predictions.json'}")
    print(f"Run summary: {RUN_DIR / 'run_summary.json'}")


if __name__ == "__main__":
    main()


Saving to: runs\GH199681_26s_bgs
Using model: C:\Users\benne\Computer_Vision\BoxingPunchDetection\yolo26_models\resume_yolo26s\weights\best.pt
Testing video: C:\Users\benne\Computer_Vision\BoxingPunchDetection\input_video\task_kam4_gh199681\data\GH199681.mp4
Saving outputs to: runs\GH199681_26s_bgs
BG gate frame=250 yielded=221 skipped=0 motion_pixels=11703
BG gate frame=500 yielded=471 skipped=0 motion_pixels=5136
BG gate frame=750 yielded=721 skipped=0 motion_pixels=12631
BG gate frame=1000 yielded=971 skipped=0 motion_pixels=9890
BG gate frame=1250 yielded=1221 skipped=0 motion_pixels=4421
BG gate frame=1500 yielded=1471 skipped=0 motion_pixels=20503
BG gate frame=1750 yielded=1721 skipped=0 motion_pixels=12152
BG gate frame=2000 yielded=1971 skipped=0 motion_pixels=9642
BG gate frame=2250 yielded=2221 skipped=0 motion_pixels=8604
BG gate frame=2500 yielded=2471 skipped=0 motion_pixels=9878
BG gate frame=2750 yielded=2721 skipped=0 motion_pixels=9069
BG gate frame=3000 yielded=2971 